# DACON 236749 · AI 음악 플레이리스트 채널의 Shorts · Windows 로컬

**Windows 10/11 x64 · Python 3.11+ · Jupyter/VS Code 노트북 · 최대 100,000개 목표**

**AI로 생성한 음악 플레이리스트를 업로드하는 채널**을 자동으로 찾고, 그 채널의 Shorts를 수집합니다. 긴 플레이리스트 영상은 채널 확인용 메타데이터만 읽습니다.

PC에 연동된 Google Drive 폴더에 오디오와 메타데이터를 저장합니다. Google 계정은 Drive 데스크톱 앱의 기존 로그인을 사용하므로 노트북에서 별도 OAuth 인증을 하지 않습니다. 원본 다운로드·SQLite 작업 DB는 로컬 작업 폴더에 두고 결과와 백업본만 연동 폴더에 기록합니다.

NVIDIA GPU 정보를 확인하는 셀을 포함했습니다. 이 파이프라인의 오디오 리샘플링·코덱 변환은 CPU FFmpeg 작업이며, GPU 모델 추론은 수행하지 않습니다. 기존 PyTorch/CUDA를 설치하거나 변경하지 않습니다.

대상 웹 폴더: [사용자가 지정한 Google Drive 폴더](https://drive.google.com/drive/folders/1bdbxE0X-51N6tvSUznLPLsWdOO-rCHv0). **이 웹 폴더에 해당하는 PC 경로를 설정하거나 폴더 선택창에서 선택하세요. 웹 폴더 ID만으로 Windows 경로를 알아낼 수는 없습니다.**

처음에는 50개 점검 모드로 실행하고 확인 후 `RUN_MODE = 'full'`로 변경합니다. 실제 수집이나 사용자의 Drive 폴더에 쓰기는 노트북을 실행할 때 시작됩니다.

## 0. 평가 데이터와 일치시키는 범위

| 항목 | 공개 조건 | 이 노트북 |
|---|---|---|
| 샘플링레이트 | 16 kHz | 모든 저장 오디오를 16 kHz로 변환 후 재디코딩 검증 |
| 길이 | 4~60초 | 4초 미만 제외, 60초 초과 Shorts는 재현 가능한 위치에서 60초 한 번 추출 |
| 채널 | 모노/스테레오 공존 | 원본의 1/2채널 유지, 2채널 초과는 2채널로 변환 |
| 파일 형식 | MP3/WAV/FLAC 등 | 기본 MP3/WAV/FLAC 혼합; 원본 ID로 결정 |
| 전화채널 | 일부 포함 | 선택적 대역 제한·8 kHz μ-law 처리; 기본 비활성화 |

실제 1,200개 평가 파일은 비공개이고 배포 샘플은 형식 확인용 더미 3개입니다. **코덱 비율·길이 분포·음량·전화채널 처리 방식까지 동일하게 재현할 근거는 없습니다.** 형식 선택의 대략 1:1:1 비율은 이 노트북의 설정이며 대회 비율이 아닙니다. 음량 정규화·무음 제거·잡음 제거·반복 패딩은 적용하지 않습니다.

출처: [대회 설명](https://dacon.io/competitions/official/236749/overview/description), [데이터 안내](https://dacon.io/competitions/official/236749/data).

## 10만 개의 의미와 라벨

- 하나의 원본 영상에서 **최대 한 개**만 저장합니다. 코덱 변경·전화음질 변환으로 수량을 늘리지 않습니다.
- 영상 ID 중복과 전체 원본을 모노 16 kHz PCM으로 디코딩한 해시의 완전일치 중복을 제거합니다. 재인코딩·일부 발췌·다른 길이의 재업로드까지 제거하는 음향 지문 시스템은 아닙니다.
- 자동 검색된 채널과 영상은 **AI 음악 후보**입니다. 키워드만으로 생성 여부·성분 존재·저작권을 확정할 수 없습니다. `WEAK_*`는 메타데이터에 의한 추정값이고, 검토하지 않은 `Y_*`는 빈칸이며 `MASK_* = 0`입니다.
- 보컬은 대회 기준상 음성입니다. 보컬과 반주가 있는 노래는 혼합 오디오입니다. AI 반주라고 해서 `VOICE_FAKE=1` 또는 `VOICE_PRESENT=0`을 넣지 않습니다.
- AI 음악 양성 후보만으로 전체 대회 학습 데이터가 완성되지는 않습니다. 실제 음악·실제/합성 음성·혼합 및 부재 사례를 별도로 확보해야 합니다.

## 외부 데이터 이용조건

대회는 최소 비영리 사용이 허용되는 공개 자원과 출처 기록을 요구합니다. 기본적으로 **영상의 Creative Commons 표시 또는 직접 기록한 이용허락 근거가 있는 경우**에만 오디오를 저장합니다. 공개 시청 가능 여부만으로 사용 권한을 가정하지 않습니다. Creative Commons 표시는 업로더의 주장으로 기록되며 원저작물 권리까지 검증하지 않습니다. 플랫폼의 자동 접근·다운로드 이용조건도 별도로 적용됩니다.

출처: [대회 규칙](https://dacon.io/competitions/official/236749/overview/rules), [YouTube 이용약관](https://www.youtube.com/static?template=terms).

## 1. 로컬 실행 환경 준비

VS Code/Jupyter에서 Python 3.11 이상 커널을 선택하고 아래부터 실행하세요. 필요한 Python 패키지를 현재 커널에 설치하고, FFmpeg가 없으면 `imageio-ffmpeg`의 실행 파일을 준비합니다. Deno가 없으면 공식 Windows 릴리스를 도구 폴더에 설치합니다. 관리자 권한이나 Linux 명령은 사용하지 않습니다.

도구와 작업 파일의 기본 위치는 `%LOCALAPPDATA%\Dacon236749`입니다. 대용량 작업용 SSD가 따로 있으면 다음 설정의 `work_dir`을 수정하세요. 패키지 갱신이 필요할 때만 `UPDATE_PACKAGES = True`로 실행하고 커널을 재시작합니다.

[FFmpeg 배포 패키지](https://github.com/imageio/imageio-ffmpeg), [Deno 공식 설치 문서](https://docs.deno.com/runtime/getting_started/installation/), [yt-dlp EJS](https://github.com/yt-dlp/yt-dlp/wiki/EJS).


In [ ]:
import sys, os, shutil, subprocess, urllib.request, zipfile, platform, importlib.util
from pathlib import Path

assert sys.version_info >= (3, 11), 'Python 3.11 이상 커널을 선택하세요.'
UPDATE_PACKAGES = False  # 기존 환경에서 yt-dlp를 업데이트하려면 True로 바꾸고 커널 재시작
requirements = {'yt_dlp': 'yt-dlp[default]', 'soundfile': 'soundfile',
                'numpy': 'numpy', 'imageio_ffmpeg': 'imageio-ffmpeg', 'filelock': 'filelock',
                'yt_dlp_ejs': 'yt-dlp[default]'}
packages = sorted(set(requirements.values() if UPDATE_PACKAGES else
    [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]))
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *(['-U'] if UPDATE_PACKAGES else []), *packages], check=True)
    importlib.invalidate_caches()

LOCAL_BASE = Path(os.environ.get('LOCALAPPDATA', str(Path.cwd() / 'work'))) / 'Dacon236749'
TOOLS_DIR = LOCAL_BASE / 'tools'
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
NO_WINDOW = subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0

import imageio_ffmpeg
FFMPEG = shutil.which('ffmpeg')
if not FFMPEG:
    bundled = Path(imageio_ffmpeg.get_ffmpeg_exe())
    executable = TOOLS_DIR / ('ffmpeg.exe' if os.name == 'nt' else 'ffmpeg')
    if not executable.exists() or executable.stat().st_size != bundled.stat().st_size:
        shutil.copy2(bundled, executable)
        if os.name != 'nt':
            executable.chmod(0o755)
    FFMPEG = str(executable)

DENO_PATH = shutil.which('deno')
if not DENO_PATH:
    DENO_PATH = str(TOOLS_DIR / ('deno.exe' if os.name == 'nt' else 'deno'))
    if not Path(DENO_PATH).exists():
        machine = platform.machine().lower()
        if os.name == 'nt' and machine in {'amd64', 'x86_64'}:
            target = 'x86_64-pc-windows-msvc'
        elif sys.platform == 'linux' and machine in {'amd64', 'x86_64'}:
            target = 'x86_64-unknown-linux-gnu'
        else:
            raise RuntimeError('이 환경에는 Deno를 공식 설치 문서에 따라 설치한 뒤 재실행하세요.')
        archive_path = TOOLS_DIR / 'deno.zip'
        urllib.request.urlretrieve(
            f'https://github.com/denoland/deno/releases/latest/download/deno-{target}.zip', archive_path)
        executable_name = 'deno.exe' if os.name == 'nt' else 'deno'
        with zipfile.ZipFile(archive_path) as archive:
            with archive.open(executable_name) as src, open(DENO_PATH, 'wb') as dst:
                shutil.copyfileobj(src, dst)
        if os.name != 'nt':
            Path(DENO_PATH).chmod(0o755)
        archive_path.unlink()

def command_output(args):
    return subprocess.check_output(args, text=True, encoding='utf-8', errors='replace',
                                   creationflags=NO_WINDOW, timeout=30).strip()

print('Python:', sys.executable, platform.python_version())
print('FFmpeg:', command_output([FFMPEG, '-version']).splitlines()[0])
deno_version = command_output([DENO_PATH, '--version'])
import re
v = re.search(r'deno (\d+)\.(\d+)\.(\d+)', deno_version)
assert v and tuple(map(int, v.groups())) >= (2, 3, 0), 'Deno 2.3.0 이상이 필요합니다.'
print('Deno:', deno_version.splitlines()[0])
GPU_INFO = 'nvidia-smi 미발견; 오디오 전처리는 CPU로 실행'
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    try:
        GPU_INFO = command_output([nvidia_smi, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
    except (subprocess.SubprocessError, OSError) as exc:
        GPU_INFO = 'GPU 정보 확인 실패: ' + str(exc)
print('NVIDIA GPU:', GPU_INFO)
print('이 수집 파이프라인의 오디오 리샘플링/코덱 변환은 CPU에서 실행합니다.')


## 2. 사용자 설정

`DRIVE_PARENT_PATH`에 **앞서 지정한 웹 폴더의 PC 경로**를 입력합니다. 빈 문자열이면 8번 셀 실행 시 폴더 선택창을 엽니다. 예시 경로는 실제 경로로 바꾸세요. 탐색기의 “경로 복사” 결과를 사용할 수 있습니다.

`work_dir`은 Google Drive 밖의 로컬 작업 폴더로 유지합니다. 기존 결과에 이어서 수집하려면 저장 경로·데이터셋 이름·전처리 설정을 유지하세요. 같은 PC의 중복 실행은 파일 잠금으로 막으며, 다른 PC에서는 같은 데이터셋을 동시에 실행하지 않습니다.

채널 미지정 시 자동으로 후보를 검색합니다. `seed_channels`, `video_permissions`, `label_overrides`는 확인한 근거가 있을 때 추가하세요. `ai_scope='all_ai_music'`에는 제작자의 채널 전체 AI 음악 생성 근거가 필요합니다. 자동 후보는 확정 라벨로 간주하지 않습니다.


채널 조건은 기본적으로 최근 20개 업로드 중 **10분 이상 플레이리스트/연속 음악 영상 2개 이상에서 AI 음악 생성 근거**가 확인되는 것입니다. 상세 설명 확인은 채널당 8개까지입니다. 제목에 AI가 없어도 설명의 음악 생성 문구·Suno/Udio 표기로 후보를 찾습니다. 이는 제목/설명 기반 선별이며, 실제 생성 이력 인증은 아닙니다.

`allow_channel_inferred_shorts=True`이면 개별 Shorts에 AI 태그가 없어도 적격 채널의 미확인 후보로 수집합니다. `ai_evidence_type='channel_playlist_inferred'`로 기록하며 확정 라벨은 채우지 않습니다. 개별 영상에도 생성 근거를 요구하려면 False로 바꾸세요. 이용허락 확인은 두 경우 모두 적용됩니다.


In [ ]:
DRIVE_PARENT_PATH = r''  # 예: r'G:\내 드라이브\AI음악'; 빈 값이면 폴더 선택창

RUN_MODE = 'pilot'  # 'pilot': 이번 실행 최대 50개 / 'full': 전체 목표 수집

CFG = {
    'drive_parent_folder_id': '1bdbxE0X-51N6tvSUznLPLsWdOO-rCHv0',
    'dataset_name': 'dacon236749_ai_playlist_shorts_v2',
    'work_dir': str(LOCAL_BASE / 'work'),
    'target_unique_videos': 100_000,
    'sample_rate': 16_000,
    'min_seconds': 4.0,
    'max_seconds': 60.0,
    'max_source_seconds': 180.0,  # 수집 상한; Shorts 여부는 /shorts 탭으로 확인
    'max_source_mb': 100,
    'formats': ['flac', 'wav', 'mp3'],  # 저장 효율 우선이면 새 데이터셋에서 ['flac']
    'channel_mode': 'preserve',  # preserve / mono / mixed(일부 모노 변환)
    'telephone_fraction': 0.0,  # 예: 0.1. 대회 실제 비율이 아닌 선택적 학습 변형
    'seed': 236749,
    'shard_size': 500,
    'checkpoint_every': 50,
    'checkpoint_min_seconds': 300,  # 평상시 백업 최소 간격; 샤드 확정 전후는 즉시 백업
    'max_candidates': 350_000,
    'max_channels': 10_000,
    'shorts_per_channel': 10_000,
    'search_results_per_query': 200,
    'playlist_min_seconds': 600,       # 10분 이상 음악 플레이리스트/연속 믹스
    'min_ai_playlist_uploads': 2,      # AI 생성 근거가 있는 업로드 2개 이상
    'channel_upload_scan_limit': 20,   # 채널의 최근 업로드 확인 범위
    'playlist_metadata_checks': 8,    # 채널당 상세 설명을 읽을 영상 수 상한
    'allow_channel_inferred_shorts': True,  # 채널 근거만 있는 Shorts도 미확인 후보로 수집
    'request_pause': 1.5,
    'download_pause': 5.0,
    'max_attempts_per_video': 3,
    'max_consecutive_errors': 5,
    'session_max_new_samples': 100_000,
    'session_max_attempts': 1_000_000,
    'session_max_seconds': 24 * 3600,
    'min_free_local_gb': 5,
    'deno_path': DENO_PATH,
    'seed_channels': [
        # {
        #   'url': 'https://www.youtube.com/@실제채널핸들',
        #   'name': '채널명',
        #   'ai_scope': 'candidate',  # 확인한 경우 'all_ai_music'
        #   'ai_evidence': '제작자의 AI 음악 생성 설명 URL 및 해당 문구',
        #   'rights_url': '',  # 대상 영상에 적용되는 이용허락 근거가 있을 때 기입
        # },
    ],
    'search_queries': [
        'AI generated music playlist', 'Suno music playlist', 'Udio music playlist',
        'AI jazz playlist', 'AI lofi playlist', 'AI chill music mix',
        'AI cafe music playlist', 'AI instrumental music compilation',
        'AI 생성 음악 플레이리스트', '수노 음악 플레이리스트',
        'AI 감성 팝송 플레이리스트', 'AI 재즈 플레이리스트',
        'AI 카페 음악 모음', 'Suno lofi playlist',
        'AI 音楽 プレイリスト', 'AI 音乐 歌单',
        '감성 팝송 플레이리스트', '카페 재즈 플레이리스트',
    ],
    'video_permissions': {
        # '실제영상ID11자': '영상별 이용허락 근거 URL 또는 문서 위치',
    },
    'label_overrides': {
        # 저장될 crop을 직접 검토한 경우에만 추가. 잘린 구간 밖의 보컬 유무를 전이하지 않음.
        # '실제영상ID11자': {
        #     'FILE_FAKE': 1, 'VOICE_FAKE': None, 'MUSIC_FAKE': 1,
        #     'VOICE_PRESENT': 0, 'MUSIC_PRESENT': 1,
        #     'evidence': '해당 crop 청취/제작자 생성 이력 근거',
        # },
    },
}
assert RUN_MODE in {'pilot', 'full'}
print('목표:', CFG['target_unique_videos'], '/ 실행 모드:', RUN_MODE)
for ch in [1, 2]:
    gib = CFG['target_unique_videos'] * 30 * 16000 * ch * 2 / 1024**3
    print(f'평균 30초, {ch}채널 PCM16 기준 약 {gib:.1f} GiB (압축/메타데이터 제외 가정)')

## 3. 공통 함수 및 Drive 연동 폴더 저장 모듈

임시 파일에 복사·파일시스템 flush·체크섬 확인 후 최종 이름으로 교체합니다. 저장 완료 수는 **PC의 연동 폴더에서 읽기 검증을 마친 수량**이며, 클라우드 업로드 완료 수량을 뜻하지 않습니다. 실제 동기화는 Drive 앱이 담당합니다.

코드의 `folder_id`는 로컬 경로의 식별자이며 Google API의 폴더 ID가 아닙니다. `shard_storage_key`는 TAR 파일 이름입니다. Google API 패키지·토큰·쿠키를 사용하지 않습니다.


In [ ]:
from __future__ import annotations
import csv
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import tarfile
import tempfile
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import soundfile as sf
import yt_dlp

TARGETS = ['FILE_FAKE', 'VOICE_FAKE', 'MUSIC_FAKE', 'VOICE_PRESENT', 'MUSIC_PRESENT']
PIPELINE_VERSION = '1.1.0-local'

def now():
    return datetime.now(timezone.utc).isoformat()

def digest_file(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def stable_number(key):
    return int(hashlib.sha256(str(key).encode()).hexdigest()[:16], 16) / 2**64

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def canonical_channel(url):
    p = urlparse(url)
    if p.scheme != 'https' or p.hostname not in {'youtube.com', 'www.youtube.com', 'm.youtube.com'}:
        raise ValueError('https://www.youtube.com/@handle 또는 /channel/UC... 주소가 필요합니다.')
    parts = p.path.strip('/').split('/')
    if parts[0].startswith('@'):
        base = parts[0]
    elif parts[0] in {'channel', 'c', 'user'} and len(parts) >= 2:
        base = '/'.join(parts[:2])
    else:
        raise ValueError(f'채널 주소가 아닙니다: {url}')
    return 'https://www.youtube.com/' + base

def validate_config(cfg):
    assert cfg['target_unique_videos'] > 0
    assert 4 <= cfg['min_seconds'] <= cfg['max_seconds'] <= 60
    assert cfg['sample_rate'] == 16000
    assert cfg['formats'] and set(cfg['formats']) <= {'flac', 'wav', 'mp3'}
    assert 0 <= cfg['telephone_fraction'] <= 1
    assert cfg['channel_mode'] in {'preserve', 'mono', 'mixed'}
    assert cfg['shard_size'] > 0 and cfg['checkpoint_every'] > 0
    assert re.fullmatch(r'[A-Za-z0-9_.-]+', cfg['dataset_name'])
    assert cfg['max_candidates'] >= cfg['target_unique_videos']
    for source in cfg['seed_channels']:
        canonical_channel(source['url'])
        if source.get('ai_scope') == 'all_ai_music' and not source.get('ai_evidence'):
            raise ValueError('all_ai_music 채널에는 ai_evidence가 필요합니다.')

In [ ]:
class SyncFolderStore:
    """Write to an existing desktop-sync folder; never claim cloud upload verification."""
    def __init__(self, parent_path, dataset_name, lock_dir=None):
        from filelock import FileLock, Timeout
        parent = Path(parent_path).expanduser().resolve(strict=True)
        if not parent.is_dir():
            raise NotADirectoryError(str(parent))
        if not re.fullmatch(r'[A-Za-z0-9_.-]+', dataset_name) or dataset_name in {'.', '..'}:
            raise ValueError('데이터셋 이름이 올바르지 않습니다.')
        self.root = parent / dataset_name
        identity = os.path.normcase(str(self.root.resolve()))
        self.folder_id = 'local-sync:' + hashlib.sha256(identity.encode()).hexdigest()
        locks = Path(lock_dir or (Path(tempfile.gettempdir()) / 'dacon236749_locks'))
        locks.mkdir(parents=True, exist_ok=True)
        self.lock = FileLock(str(locks / (hashlib.sha256(identity.encode()).hexdigest() + '.lock')))
        try:
            self.lock.acquire(timeout=0)
        except Timeout as exc:
            raise RuntimeError('이 PC에서 같은 데이터셋을 다른 커널이 사용 중입니다. 먼저 해당 커널을 종료하세요.') from exc
        try:
            self.root.mkdir(exist_ok=True)
            marker = self.root / '.dataset_identity.json'
            if marker.exists():
                old = json.loads(marker.read_text(encoding='utf-8'))
                if old.get('backend') != 'desktop_sync' or old.get('dataset_name') != dataset_name:
                    raise RuntimeError('기존 폴더의 데이터셋 식별 정보와 다릅니다.')
            else:
                self._write_bytes(marker, json.dumps({'backend': 'desktop_sync', 'dataset_name': dataset_name,
                    'created_at': now(), 'cloud_sync_verified': False}).encode())
        except BaseException:
            self.close()
            raise
        print('연동 폴더 저장 위치:', self.root)
        print('클라우드 동기화 완료 여부는 Google Drive 데스크톱 앱에서 확인하세요.')

    def close(self):
        self.lock.release()

    def _path(self, name):
        if not isinstance(name, str) or name in {'', '.', '..'} or Path(name).name != name or '/' in name or '\\' in name:
            raise ValueError('저장 키에는 파일 이름만 사용할 수 있습니다.')
        path = self.root / name
        if not path.resolve().is_relative_to(self.root.resolve()):
            raise ValueError('저장 경로가 데이터셋 폴더 밖을 가리킵니다.')
        return path

    @staticmethod
    def _replace(part, destination):
        # Antivirus/DriveFS may hold a brief Windows file handle.
        for attempt in range(5):
            try:
                os.replace(part, destination)
                return
            except PermissionError:
                if attempt == 4:
                    raise
                time.sleep(min(8, 2**attempt))

    @classmethod
    def _write_bytes(cls, destination, data):
        import uuid
        part = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
        try:
            with part.open('xb') as f:
                f.write(data)
                f.flush()
                os.fsync(f.fileno())
            cls._replace(part, destination)
        finally:
            part.unlink(missing_ok=True)

    @classmethod
    def _copy(cls, source, destination):
        import uuid
        part = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
        expected = digest_file(source, 'md5')
        try:
            with Path(source).open('rb') as src, part.open('xb') as dst:
                shutil.copyfileobj(src, dst, length=4 * 1024**2)
                dst.flush()
                os.fsync(dst.fileno())
            if digest_file(part, 'md5') != expected:
                raise IOError('복사된 임시 파일의 체크섬이 다릅니다.')
            cls._replace(part, destination)
            if digest_file(destination, 'md5') != expected:
                raise IOError('연동 폴더 파일의 체크섬이 다릅니다.')
        finally:
            part.unlink(missing_ok=True)

    def stat(self, file_id):
        path = self._path(file_id)
        if not path.exists():
            return None
        if not path.is_file():
            raise IsADirectoryError(str(path))
        return {'id': path.name, 'name': path.name, 'size': str(path.stat().st_size),
                'md5Checksum': digest_file(path, 'md5')}

    def find(self, name):
        return self.stat(name)

    def put(self, path, name=None, file_id=None):
        path = Path(path)
        key = file_id or name or path.name
        destination = self._path(key)
        if not path.is_file():
            raise FileNotFoundError(str(path))
        if not self.root.is_dir():
            raise FileNotFoundError('Drive 연동 폴더 연결이 해제되었습니다.')
        if destination.is_file():
            previous = self.stat(key)
            if previous['md5Checksum'] == digest_file(path, 'md5'):
                return previous
            if key.endswith('.tar'):
                raise IOError('기존 오디오 샤드와 내용이 다릅니다. 덮어쓰지 않습니다: ' + key)
        if self.free_bytes() < path.stat().st_size + 64 * 1024**2:
            raise OSError('대상 파일시스템의 여유 공간이 부족합니다.')
        self._copy(path, destination)
        return self.stat(key)

    def get(self, file_id, path):
        self._copy(self._path(file_id), Path(path))

    def free_bytes(self):
        # Filesystem-reported space; NOT a verified Google Account cloud quota.
        return shutil.disk_usage(self.root).free

## 4. 체크포인트와 중단·재개

SQLite 작업 DB는 로컬 SSD에서 사용합니다. Drive 연동 폴더에는 닫힌 백업 파일만 복사하며, 평상시 5분 간격과 샤드 저장 전후에 체크포인트를 갱신합니다. 같은 PC에서 재개하면 최신 로컬 DB를 유지하고, 로컬 DB가 없을 때는 연동 폴더의 백업본을 복원합니다.

저장 전 샤드 이름과 체크섬을 남기므로 파일 복사 직후 중단되더라도 다음 실행에서 기존 파일을 확인하고 완료 처리할 수 있습니다. 결과와 체크포인트의 연동 폴더 쓰기가 검증된 뒤 임시 오디오를 정리합니다.

Google Drive의 클라우드 동기화는 비동기입니다. 다른 PC에서 복원하거나 PC를 종료하기 전에는 Drive 앱에서 동기화 완료 및 오류 여부를 확인하세요. 스트리밍 캐시/로컬 디스크 여유와 클라우드 계정 저장 용량은 별개입니다. [Drive 데스크톱 안내](https://support.google.com/drive/answer/10838124).


In [ ]:
SCHEMA = '''
CREATE TABLE IF NOT EXISTS config (key TEXT PRIMARY KEY, value TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS queries (query TEXT PRIMARY KEY, status TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS channels (
 url TEXT PRIMARY KEY, channel_id TEXT, title TEXT, evidence TEXT,
 ai_scope TEXT, rights_url TEXT, source TEXT, enumerated_limit INTEGER DEFAULT 0,
 status TEXT DEFAULT 'pending', error TEXT);
CREATE TABLE IF NOT EXISTS videos (
 id TEXT PRIMARY KEY, channel_url TEXT NOT NULL, title TEXT, url TEXT NOT NULL,
 status TEXT NOT NULL DEFAULT 'pending', attempts INTEGER NOT NULL DEFAULT 0,
 error TEXT, discovered_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS samples (
 video_id TEXT PRIMARY KEY, pcm_sha256 TEXT NOT NULL UNIQUE,
 path TEXT NOT NULL, meta TEXT NOT NULL, shard_seq INTEGER);
CREATE TABLE IF NOT EXISTS shards (
 seq INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE, file_id TEXT NOT NULL,
 md5 TEXT, size INTEGER, status TEXT NOT NULL);
CREATE INDEX IF NOT EXISTS video_status ON videos(status, attempts);
CREATE INDEX IF NOT EXISTS sample_shard ON samples(shard_seq);
'''

class Dataset:
    def __init__(self, cfg, store):
        validate_config(cfg)
        self.cfg, self.store = cfg, store
        self.root = (Path(cfg['work_dir']) / cfg['dataset_name']).resolve()
        sync_root = store.root.resolve()
        if self.root.is_relative_to(sync_root) or sync_root.is_relative_to(self.root):
            raise ValueError('작업 폴더와 Drive 저장 폴더는 서로 포함되지 않는 별도 경로여야 합니다.')
        self._last_checkpoint = 0.0
        self.root.mkdir(parents=True, exist_ok=True)
        self.raw = self.root / 'raw'
        self.stage = self.root / 'staged'
        self.raw.mkdir(exist_ok=True)
        self.stage.mkdir(exist_ok=True)
        self.db_path = self.root / 'state.sqlite'
        if not self.db_path.exists():
            remote = store.find('state.sqlite')
            if remote:
                store.get(remote['id'], self.db_path)
                print('Drive 체크포인트 복원 완료')
        self.db = sqlite3.connect(self.db_path)
        self.db.row_factory = sqlite3.Row
        self.db.executescript(SCHEMA)
        self.db.execute('PRAGMA journal_mode=DELETE')
        self.db.execute('PRAGMA synchronous=FULL')
        profile_keys = ['sample_rate', 'min_seconds', 'max_seconds', 'formats',
                        'telephone_fraction', 'channel_mode', 'seed']
        profile = {'pipeline': PIPELINE_VERSION, 'folder_id': store.folder_id,
                   **{k: cfg[k] for k in profile_keys}}
        encoded = json.dumps(profile, sort_keys=True)
        old = self.db.execute("SELECT value FROM config WHERE key='profile'").fetchone()
        if old and old[0] != encoded:
            raise ValueError('기존 데이터셋의 전처리 설정과 다릅니다. dataset_name을 변경하세요.')
        self.db.execute("INSERT OR IGNORE INTO config VALUES ('profile', ?)", (encoded,))
        self.db.commit()
        self.recover()

    def checkpoint(self, force=False):
        self.db.commit()
        if not force and time.monotonic() - self._last_checkpoint < self.cfg.get('checkpoint_min_seconds', 300):
            return
        snapshot = self.root / 'state.snapshot.sqlite'
        with sqlite3.connect(snapshot) as backup:
            self.db.backup(backup)
        self.store.put(snapshot, 'state.sqlite')
        self._last_checkpoint = time.monotonic()

    def status(self):
        return dict(self.db.execute('SELECT status,COUNT(*) FROM videos GROUP BY status').fetchall())

    def total_samples(self):
        return self.db.execute('SELECT COUNT(*) FROM samples').fetchone()[0]

    def recover(self):
        # A keyboard interrupt can checkpoint a partly assembled local tar.
        for shard in self.db.execute("SELECT seq FROM shards WHERE status='building'").fetchall():
            self.db.execute('UPDATE samples SET shard_seq=NULL WHERE shard_seq=?', (shard['seq'],))
            self.db.execute('DELETE FROM shards WHERE seq=?', (shard['seq'],))
        # Reconcile an uploaded shard even if the final checkpoint never ran.
        for shard in self.db.execute("SELECT * FROM shards WHERE status='uploading'").fetchall():
            remote = self.store.stat(shard['file_id'])
            if remote:
                if remote.get('md5Checksum') != shard['md5']:
                    raise IOError('복구 대상 shard 체크섬 불일치; 원본을 보존하고 중단합니다.')
                self.mark_committed(shard['seq'])
            else:
                local = self.root / shard['name']
                if local.exists() and digest_file(local, 'md5') == shard['md5']:
                    self.store.put(local, file_id=shard['file_id'])
                    self.mark_committed(shard['seq'])
                else:
                    ids = [r[0] for r in self.db.execute(
                        'SELECT video_id FROM samples WHERE shard_seq=?', (shard['seq'],))]
                    self.db.executemany("UPDATE videos SET status='pending', attempts=0 WHERE id=?", [(v,) for v in ids])
                    self.db.execute('DELETE FROM samples WHERE shard_seq=?', (shard['seq'],))
                    self.db.execute('DELETE FROM shards WHERE seq=?', (shard['seq'],))
        for row in self.db.execute('SELECT video_id,path FROM samples WHERE shard_seq IS NULL').fetchall():
            if not Path(row['path']).exists():
                self.db.execute('DELETE FROM samples WHERE video_id=?', (row['video_id'],))
                self.db.execute("UPDATE videos SET status='pending',attempts=0 WHERE id=?", (row['video_id'],))
        self.db.execute("UPDATE videos SET status='pending' WHERE status='processing'")
        self.db.commit()
        self.checkpoint()
        self.clean_committed()

    def mark_committed(self, seq):
        self.db.execute("UPDATE shards SET status='committed' WHERE seq=?", (seq,))
        self.db.execute("UPDATE videos SET status='done',error=NULL WHERE id IN "
                        '(SELECT video_id FROM samples WHERE shard_seq=?)', (seq,))
        self.db.commit()

    def clean_committed(self, seq=None):
        query = "SELECT s.path FROM samples s JOIN shards h ON s.shard_seq=h.seq WHERE h.status='committed'"
        rows = self.db.execute(query + (' AND h.seq=?' if seq is not None else ''), (seq,) if seq is not None else ())
        for row in rows:
            path = Path(row[0]).resolve()
            if path.is_relative_to(self.stage.resolve()):
                path.unlink(missing_ok=True)
        query = "SELECT name FROM shards WHERE status='committed'"
        for row in self.db.execute(query + (' AND seq=?' if seq is not None else ''), (seq,) if seq is not None else ()):
            (self.root / row[0]).unlink(missing_ok=True)

    def flush(self):
        while True:
            rows = self.db.execute('SELECT * FROM samples WHERE shard_seq IS NULL ORDER BY video_id LIMIT ?',
                                   (self.cfg['shard_size'],)).fetchall()
            if not rows:
                self.checkpoint(force=True)
                return
            file_id = ''
            seq = self.db.execute("INSERT INTO shards(file_id,status) VALUES (?, 'building')", (file_id,)).lastrowid
            name = f'audio-{seq:06d}.tar'
            file_id = name
            tar_path = self.root / name
            # Tar keeps many tiny Drive writes out of the data path.
            with tarfile.open(tar_path, 'w') as archive:
                for row in rows:
                    meta = json.loads(row['meta'])
                    meta['shard'] = name
                    meta['shard_storage_key'] = file_id
                    archive.add(row['path'], arcname=meta['audio_member'], recursive=False)
                    payload = json.dumps(meta, ensure_ascii=False, allow_nan=False).encode('utf-8')
                    item = tarfile.TarInfo(f"metadata/{row['video_id']}.json")
                    item.size = len(payload)
                    archive.addfile(item, io.BytesIO(payload))
                    self.db.execute('UPDATE samples SET shard_seq=?,meta=? WHERE video_id=?',
                                    (seq, json.dumps(meta, ensure_ascii=False), row['video_id']))
            self.db.execute("UPDATE shards SET name=?,file_id=?,md5=?,size=?,status='uploading' WHERE seq=?",
                            (name, file_id, digest_file(tar_path, 'md5'), tar_path.stat().st_size, seq))
            self.db.commit()
            self.checkpoint(force=True)  # Save recovery metadata before copying the tar.
            self.store.put(tar_path, file_id=file_id)
            self.mark_committed(seq)
            self.checkpoint(force=True)  # Delete scratch only after verified sync-folder writes.
            self.clean_committed(seq)
            print(f'연동 폴더 저장 완료: {name} / 누적 {self.status().get("done", 0):,}개')

    def retry_errors(self):
        self.db.execute("UPDATE videos SET status='pending', attempts=0, error=NULL WHERE status='error'")
        self.db.execute("UPDATE queries SET status='pending' WHERE status='error'")
        self.db.execute("UPDATE channels SET status='pending' WHERE status='error'")
        self.db.commit()
        self.checkpoint()

    def reconsider_skipped(self):
        # Use after adding new rights evidence or changing candidate filters.
        self.db.execute("UPDATE videos SET status='pending', attempts=0, error=NULL WHERE status='skipped'")
        self.db.commit()
        self.checkpoint()

## 5. AI 음악 플레이리스트 채널 탐색 → Shorts 수집

1. 음악 플레이리스트 검색어로 긴 음악 업로드의 채널을 찾습니다.
2. 해당 채널의 최근 업로드에서 플레이리스트 형식·음악 생성 근거를 확인합니다.
3. 조건을 충족하는 채널만 `/shorts` 탭을 열어 영상 링크를 수집합니다.
4. Shorts 오디오만 다운로드합니다. 긴 플레이리스트를 잘라 수량을 채우지는 않습니다.

제작 강좌·사용법·부업·수익화 설명 제목은 제외합니다. AI 이미지/영상 제작 언급만으로는 AI 음악 근거로 인정하지 않습니다. 채널 확인 결과와 대표 플레이리스트의 URL·설명은 `channel_screening.csv`에 기록합니다. 키워드 기반 판단의 누락/오탐은 있을 수 있습니다.

공개 검색에서 제작자가 모든 트랙을 Suno로 생성한다고 소개한 [SUNO MUSIC PLAYLIST](https://www.youtube.com/channel/UCVcLVYbArsH9mBpembkbxUw) 같은 형태가 탐색 대상입니다. 이 링크를 확정 학습 데이터나 이용허락으로 사용하지 않으며, 현재 Shorts 보유 여부와 업로드 조건은 실행 시 별도로 확인합니다.


In [ ]:
AI_RX = re.compile(r'\b(?:ai|suno|udio)\b|ai[- _]?(?:generated|music|song)|인공지능|AI생성|生成AI', re.I)
MUSIC_RX = re.compile(r'music|song|instrumental|soundtrack|beat|lofi|lo-fi|음악|노래|작곡|音楽|音乐', re.I)
EXCLUDE_RX = re.compile(r'tutorial|how to|review|news|강의|강좌|사용법|만드는\s?법|튜토리얼', re.I)
BLOCK_RX = re.compile(r'429|too many requests|confirm you.?re not a bot|sign in to confirm|captcha|403: Forbidden|HTTP Error 403', re.I)

class StopCollection(RuntimeError):
    pass

class SkipVideo(RuntimeError):
    pass

class QuietLogger:
    def debug(self, message):
        pass
    def warning(self, message):
        if 'javascript' in message.lower() or 'challenge' in message.lower():
            print(str(message)[:400])
    def error(self, message):
        pass

def ydl_options(cfg, **extra):
    options = dict(
        quiet=True, no_warnings=False, logger=QuietLogger(),
        socket_timeout=30, retries=2, fragment_retries=2, extractor_retries=2,
        concurrent_fragment_downloads=1, sleep_interval_requests=cfg['request_pause'],
        sleep_interval=cfg['download_pause'], max_sleep_interval=cfg['download_pause'] + 2,
        ignoreerrors=False, nocheckcertificate=False, ffmpeg_location=FFMPEG,
        js_runtimes={'deno': {'path': cfg['deno_path']}},
    )
    options.update(extra)
    return options

def ai_music_evidence(text):
    return ai_audio_claim(text)

def handle_network_error(exc):
    if BLOCK_RX.search(str(exc)):
        raise StopCollection('YouTube 접근 제한 감지. 체크포인트를 저장하고 중단합니다. '
                             '현재 접근 가능한 환경과 이용조건을 확인한 뒤 재개하세요.') from exc

PLAYLIST_RX = re.compile(r'play\s?list|플레이\s?리스트|플리|모음|compilation|(?:music|jazz|lofi|chill)\s+mix|再生リスト|歌单', re.I)
PLAYLIST_MUSIC_RX = re.compile(r'music|song|track|jazz|lofi|lo-fi|ambient|piano|instrumental|bgm|음악|노래|재즈|피아노|팝송|音楽|音乐', re.I)
CHANNEL_TUTORIAL_RX = re.compile(r'tutorial|how\s+to|monetiz|make\s+money|강의|강좌|사용법|만드는\s?법|튜토리얼|수익화|부업|운영\s?노하우|채널\s?만들', re.I)
NON_AI_AUDIO_RX = re.compile(r'(?:no|not|without|non[- ])\s*ai[- ]*(?:generated\s*)?(?:music|songs?|audio)|(?:음악|음원|노래)[^.!?\n]{0,35}AI[^.!?\n]{0,12}(?:아닙|않았|않습)', re.I)
GENERATOR_AUDIO_RX = re.compile(r'\b(?:suno|udio)\b|수노|수노AI', re.I)
AI_AUDIO_RX = re.compile(
    r'\bai[- ]+(?:music|songs?|tracks?|audio)\b|AI\s*(?:음악|작곡)|'
    r'(?:ai|인공지능)[- ]*(?:generated|composed|created|produced)[- ]*(?:music|songs?|tracks?|audio)|'
    r'(?:music|songs?|tracks?|audio)[^.!?\n]{0,70}(?:generated|composed|created|produced)[^.!?\n]{0,35}\bai\b|'
    r'(?:음악|음원|노래|곡)[^.!?\n]{0,40}(?:AI|인공지능)[^.!?\n]{0,30}(?:생성|제작|만들|작곡)|'
    r'(?:AI|인공지능)[^.!?\n]{0,35}(?:생성|제작|만든|작곡)[^.!?\n]{0,20}(?:음악|음원|노래|곡)', re.I)

def ai_audio_claim(text):
    text = text or ''
    if NON_AI_AUDIO_RX.search(text):
        return False
    return bool(AI_AUDIO_RX.search(text) or GENERATOR_AUDIO_RX.search(text))

def playlist_upload_candidate(info, cfg):
    title = str(info.get('title') or '')
    if CHANNEL_TUTORIAL_RX.search(title):
        return False
    duration = info.get('duration')
    if duration is not None and duration < cfg['playlist_min_seconds']:
        return False
    return bool(PLAYLIST_RX.search(title) or
                (duration is not None and duration >= cfg['playlist_min_seconds'] and PLAYLIST_MUSIC_RX.search(title)))

def screening_profile(cfg):
    return json.dumps({k: cfg[k] for k in ['playlist_min_seconds', 'min_ai_playlist_uploads',
        'channel_upload_scan_limit', 'playlist_metadata_checks']}, sort_keys=True)

def ensure_screening_table(ds):
    ds.db.execute('''CREATE TABLE IF NOT EXISTS channel_screening (
        url TEXT NOT NULL, profile TEXT NOT NULL, status TEXT NOT NULL,
        evidence TEXT NOT NULL, checked_at TEXT NOT NULL, PRIMARY KEY(url,profile))''')
    ds.db.commit()

def screen_playlist_channel(ds, url):
    cfg = ds.cfg
    profile = screening_profile(cfg)
    cached = ds.db.execute('SELECT status,evidence FROM channel_screening WHERE url=? AND profile=?',
                          (url, profile)).fetchone()
    if cached and cached['status'] != 'error':
        evidence = json.loads(cached['evidence'])
        return (evidence if cached['status'] == 'candidate' else None)
    with yt_dlp.YoutubeDL(ydl_options(cfg, extract_flat='in_playlist', skip_download=True,
            playlistend=cfg['channel_upload_scan_limit'])) as ydl:
        listing = ydl.extract_info(url + '/videos', download=False)
    if not listing:
        raise RuntimeError('채널 업로드 목록을 읽을 수 없습니다.')
    channel_description = str(listing.get('description') or '')[:12000]
    entries = list(listing.get('entries') or [])
    evidence = {'discovery_mode': 'ai_music_playlist_uploads',
                'channel_id': listing.get('channel_id'), 'channel_title': listing.get('channel'),
                'channel_description': channel_description, 'playlist_examples': [],
                'note': '제작자 메타데이터 기반 채널 후보; 개별 Shorts AI 여부/이용허락 미확정'}
    seen = set()
    checked = 0
    for entry in entries:
        if not entry or not playlist_upload_candidate(entry, cfg):
            continue
        vid = str(entry.get('id') or '')
        if not re.fullmatch(r'[A-Za-z0-9_-]{11}', vid) or vid in seen:
            continue
        seen.add(vid)
        if checked >= cfg['playlist_metadata_checks']:
            break
        checked += 1
        # Fetch metadata only; playlist audio is never downloaded here.
        with yt_dlp.YoutubeDL(ydl_options(cfg, noplaylist=True, skip_download=True)) as ydl:
            item = ydl.extract_info('https://www.youtube.com/watch?v=' + vid, download=False)
        if not item or not playlist_upload_candidate(item, cfg):
            continue
        if item.get('duration') is None or item['duration'] < cfg['playlist_min_seconds']:
            continue
        text = str(item.get('title') or '') + '\n' + str(item.get('description') or '')
        if NON_AI_AUDIO_RX.search(text):
            continue
        own_claim = ai_audio_claim(text)
        channel_claim = ai_audio_claim(channel_description)
        if not (own_claim or channel_claim):
            continue
        evidence['playlist_examples'].append({
            'id': vid, 'url': 'https://www.youtube.com/watch?v=' + vid,
            'title': item.get('title'), 'duration': item['duration'],
            'description': str(item.get('description') or '')[:12000],
            'ai_basis': 'playlist_video_metadata' if own_claim else 'channel_description'})
        if len(evidence['playlist_examples']) >= cfg['min_ai_playlist_uploads']:
            break
    status = 'candidate' if len(evidence['playlist_examples']) >= cfg['min_ai_playlist_uploads'] else 'no_match'
    ds.db.execute('INSERT OR REPLACE INTO channel_screening VALUES (?,?,?,?,?)',
                  (url, profile, status, json.dumps(evidence, ensure_ascii=False), now()))
    ds.db.commit()
    return evidence if status == 'candidate' else None

def discover_channels(ds):
    cfg = ds.cfg
    ensure_screening_table(ds)

    def consider(url, seed=None):
        url = canonical_channel(url)
        try:
            evidence = screen_playlist_channel(ds, url)
        except Exception as exc:
            ds.db.execute('INSERT OR REPLACE INTO channel_screening VALUES (?,?,?,?,?)',
                (url, screening_profile(cfg), 'error', json.dumps({'error':str(exc)[:1800]}), now()))
            ds.db.commit()
            handle_network_error(exc)
            print('채널 확인 실패:', url, str(exc)[:160])
            return
        if not evidence:
            return
        existing = ds.db.execute('SELECT source FROM channels WHERE url=?', (url,)).fetchone()
        if seed is None and existing and existing['source'] == 'seed':
            return  # Keep user-provided rights and creator scope when search finds the same channel.
        seed = seed or {}
        if seed.get('ai_evidence'):
            evidence['user_ai_evidence'] = seed['ai_evidence']
        scope = 'all_ai_music' if seed.get('ai_scope') == 'all_ai_music' else 'ai_playlist_candidate'
        ds.db.execute('''INSERT INTO channels(url,channel_id,title,evidence,ai_scope,rights_url,source)
            VALUES (?,?,?,?,?,?,?) ON CONFLICT(url) DO UPDATE SET
            channel_id=excluded.channel_id,title=excluded.title,evidence=excluded.evidence,
            ai_scope=excluded.ai_scope,rights_url=excluded.rights_url,source=excluded.source''',
            (url, evidence.get('channel_id'), seed.get('name') or evidence.get('channel_title'),
             json.dumps(evidence, ensure_ascii=False), scope, seed.get('rights_url',''), 'seed' if seed else 'playlist_search'))
        ds.db.commit()
        print('AI 음악 플레이리스트 채널 후보:', evidence.get('channel_title') or url)

    for seed in cfg['seed_channels']:
        consider(seed['url'], seed)
        ds.checkpoint()
    for query in cfg['search_queries']:
        key = f"playlist-v2:{screening_profile(cfg)}:{cfg['search_results_per_query']}:{query}"
        old = ds.db.execute('SELECT status FROM queries WHERE query=?', (key,)).fetchone()
        if old and old[0] == 'done':
            continue
        if ds.db.execute('SELECT COUNT(*) FROM channels').fetchone()[0] >= cfg['max_channels']:
            break
        ds.db.execute("INSERT OR REPLACE INTO queries VALUES (?, 'pending')", (key,))
        capped = False
        try:
            with yt_dlp.YoutubeDL(ydl_options(cfg, extract_flat=True, skip_download=True)) as ydl:
                result = ydl.extract_info(f"ytsearch{cfg['search_results_per_query']}:{query}", download=False)
            for entry in (result or {}).get('entries', []):
                if not entry or not playlist_upload_candidate(entry, cfg):
                    continue
                channel_id = entry.get('channel_id')
                url = ('https://www.youtube.com/channel/' + channel_id) if channel_id else entry.get('channel_url')
                if url:
                    consider(url)
                    ds.checkpoint()
                if ds.db.execute('SELECT COUNT(*) FROM channels').fetchone()[0] >= cfg['max_channels']:
                    capped = True
                    break
            ds.db.execute('UPDATE queries SET status=? WHERE query=?', ('pending' if capped else 'done', key))
        except Exception as exc:
            ds.db.execute("UPDATE queries SET status='error' WHERE query=?", (key,))
            handle_network_error(exc)
            print(f'검색 실패 [{query}]: {str(exc)[:200]}')
        finally:
            ds.checkpoint()
    cursor = ds.db.execute('SELECT * FROM channel_screening ORDER BY url')
    path = ds.root / 'channel_screening.csv'
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([c[0] for c in cursor.description])
        writer.writerows(cursor)
    ds.store.put(path)

def refresh_playlist_discovery(ds):
    ensure_screening_table(ds)
    ds.db.execute("DELETE FROM channel_screening WHERE status != 'candidate'")
    ds.db.execute("UPDATE queries SET status='pending' WHERE query LIKE 'playlist-v2:%'")
    ds.db.commit()
    ds.checkpoint(force=True)


def enumerate_shorts(ds):
    cfg = ds.cfg
    n = ds.db.execute('SELECT COUNT(*) FROM videos').fetchone()[0]
    channels = ds.db.execute('SELECT * FROM channels ORDER BY CASE source WHEN \'seed\' THEN 0 ELSE 1 END,url').fetchall()
    for channel in channels:
        if n >= cfg['max_candidates']:
            break
        if channel['status'] == 'done' and channel['enumerated_limit'] >= cfg['shorts_per_channel']:
            continue
        try:
            with yt_dlp.YoutubeDL(ydl_options(cfg, extract_flat='in_playlist', skip_download=True,
                    lazy_playlist=True, playlistend=cfg['shorts_per_channel'])) as ydl:
                info = ydl.extract_info(channel['url'] + '/shorts', download=False)
                if not info:
                    raise RuntimeError('Shorts 탭 응답 없음')
                # No duration-based Shorts inference: membership in /shorts is the provenance.
                actual_url = info.get('webpage_url') or info.get('original_url') or ''
                if '/shorts' not in urlparse(actual_url).path:
                    raise RuntimeError('Shorts 탭에서 다른 페이지로 이동하여 수집을 건너뜁니다.')
                description = (info.get('description') or '')[:12000]
                evidence = json.dumps({'search_evidence': channel['evidence'],
                    'channel_description': description, 'channel_metadata_ai_music': ai_music_evidence(description)}, ensure_ascii=False)
                ds.db.execute('UPDATE channels SET channel_id=?,title=?,evidence=? WHERE url=?',
                    (info.get('channel_id') or channel['channel_id'], info.get('channel') or channel['title'], evidence, channel['url']))
                finished = True
                for index, entry in enumerate(info.get('entries') or []):
                    if not entry:
                        continue
                    vid = entry.get('id', '')
                    if not re.fullmatch(r'[A-Za-z0-9_-]{11}', vid):
                        continue
                    count = ds.db.execute('''INSERT OR IGNORE INTO videos
                        (id,channel_url,title,url,discovered_at) VALUES (?,?,?,?,?)''',
                        (vid, channel['url'], entry.get('title'), 'https://www.youtube.com/shorts/' + vid, now())).rowcount
                    if channel['source'] == 'seed':
                        # A later @handle grant must also reach earlier /channel/UC... candidates.
                        ds.db.execute("UPDATE videos SET channel_url=? WHERE id=? AND status IN ('pending','skipped','error')",
                                      (channel['url'], vid))
                    n += count
                    if index and index % 1000 == 0:
                        ds.checkpoint()
                    if n >= cfg['max_candidates']:
                        finished = False
                        break
                if finished:
                    ds.db.execute("UPDATE channels SET status='done',enumerated_limit=?,error=NULL WHERE url=?",
                                  (cfg['shorts_per_channel'], channel['url']))
        except Exception as exc:
            ds.db.execute("UPDATE channels SET status='error',error=? WHERE url=?", (str(exc)[:1500], channel['url']))
            handle_network_error(exc)
            print(f'채널 목록 실패: {channel["url"]} / {str(exc)[:160]}')
        finally:
            ds.checkpoint()
        print(f'Shorts 후보 {n:,}개 / {channel["title"]}')

def license_evidence(info, channel, cfg):
    license_text = str(info.get('license') or '')
    vid = info['id']
    if cfg['video_permissions'].get(vid):
        return {'kind': 'user_documented_permission', 'evidence': cfg['video_permissions'][vid], 'license': license_text}
    if channel['rights_url']:
        return {'kind': 'channel_permission', 'evidence': channel['rights_url'], 'license': license_text}
    if 'creative commons' in license_text.lower():
        return {'kind': 'declared_creative_commons', 'evidence': 'https://www.youtube.com/watch?v=' + vid, 'license': license_text}
    raise SkipVideo('rights_unverified: Creative Commons 표시 또는 문서화된 이용허락 없음')

def download_one(ds, row, folder):
    cfg = ds.cfg
    channel = ds.db.execute('SELECT * FROM channels WHERE url=?', (row['channel_url'],)).fetchone()
    with yt_dlp.YoutubeDL(ydl_options(cfg, noplaylist=True, format='bestaudio/best',
            max_filesize=cfg['max_source_mb'] * 1024**2,
            outtmpl=str(folder / '%(id)s.%(ext)s'), continuedl=True,
            fixup='never', cachedir=False)) as ydl:
        info = ydl.extract_info(row['url'], download=False)
        if not info or info.get('id') != row['id']:
            raise SkipVideo('video_id_mismatch')
        if info.get('is_live') or info.get('live_status') in {'is_live', 'is_upcoming'}:
            raise SkipVideo('live_video')
        duration = info.get('duration')
        if duration is None or not cfg['min_seconds'] <= duration <= cfg['max_source_seconds']:
            raise SkipVideo('source_duration_out_of_bounds')
        title = str(info.get('title') or '')
        text = title + '\n' + str(info.get('description') or '')
        if EXCLUDE_RX.search(title):
            raise SkipVideo('tutorial_or_review_title')
        # A reviewed creator-wide assertion may cover videos without repeated tags.
        creator_scope = channel['ai_scope'] == 'all_ai_music' and bool(channel['evidence'])
        playlist_inferred = (cfg.get('allow_channel_inferred_shorts', True)
                             and channel['ai_scope'] == 'ai_playlist_candidate')
        if NON_AI_AUDIO_RX.search(text):
            raise SkipVideo('explicit_non_ai_audio_statement')
        if not creator_scope and not playlist_inferred and not ai_music_evidence(text):
            raise SkipVideo('no_video_or_playlist_channel_ai_music_evidence')
        rights = license_evidence(info, channel, cfg)
        info = ydl.process_ie_result(info, download=True)
        candidates = [Path(x['filepath']) for x in info.get('requested_downloads', []) if x.get('filepath')]
        candidates.append(Path(ydl.prepare_filename(info)))
        source = next((x for x in candidates if x.is_file()), None)
        if source is None:
            raise RuntimeError('다운로드 파일이 없습니다. 파일 크기 제한 또는 yt-dlp 로그를 확인하세요.')
        if source.stat().st_size > cfg['max_source_mb'] * 1024**2:
            raise SkipVideo('source_too_large')
    metadata = {k: info.get(k) for k in ['id', 'title', 'description', 'channel', 'channel_id',
        'channel_url', 'uploader', 'upload_date', 'duration', 'license', 'tags', 'categories']}
    metadata['description'] = (metadata['description'] or '')[:16000]
    metadata.update(source_url=row['url'], shorts_tab=row['channel_url'] + '/shorts',
        channel_evidence=channel['evidence'], rights=rights,
        ai_evidence_type=('creator_scope' if creator_scope else 'video_audio_claim' if ai_music_evidence(text)
                          else 'channel_playlist_inferred'), collected_at=now())
    return source, metadata

## 6. 오디오 전처리 및 라벨 기록 모듈

저장 파일을 다시 디코딩하여 16 kHz·1/2채널·4~60초인지 검사합니다. 디지털 완전 무음/비정상 값은 제외하고 RMS와 클리핑 비율은 품질 확인용으로만 기록합니다.

`Y_*`는 학습 타깃, `MASK_*`는 타깃 확인 여부입니다. 누락 타깃을 0으로 채우면 안 됩니다. `WEAK_*`는 후보 수집 근거로부터의 추정이며 검증용 정답으로 쓰면 안 됩니다. 대회 출력의 `*_PROB`는 모델이 예측하는 확률로, 이 수집 노트북이 정답처럼 만들어 제출하지 않습니다.

In [ ]:
def ffmpeg_run(args, capture=False):
    command = [FFMPEG, '-hide_banner', '-loglevel', 'error', '-nostdin', '-y', *map(str, args)]
    result = subprocess.run(command, check=False, stdout=subprocess.PIPE if capture else subprocess.DEVNULL,
                            stderr=subprocess.PIPE, timeout=300, creationflags=subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0)
    if result.returncode:
        raise RuntimeError('FFmpeg: ' + result.stderr.decode(errors='replace')[-1600:])
    return result.stdout if capture else None

def decode_audio(path, temporary, max_seconds=None):
    args = ['-i', path, '-map', '0:a:0', '-vn']
    if max_seconds is not None:
        args += ['-t', max_seconds]
    ffmpeg_run(args + ['-c:a', 'pcm_s16le', temporary])
    audio, rate = sf.read(temporary, dtype='float32', always_2d=True)
    return audio, rate

def labels_for(vid, cfg):
    entry = cfg['label_overrides'].get(vid, {})
    if entry and not entry.get('evidence'):
        raise ValueError('확정 라벨에는 해당 클립을 검토한 evidence를 기록해야 합니다.')
    labels = {name: entry.get(name) for name in TARGETS}
    if any(value not in (None, 0, 1) for value in labels.values()):
        raise ValueError('라벨은 0, 1, None만 가능합니다.')
    for comp in ['VOICE', 'MUSIC']:
        if labels[comp + '_PRESENT'] == 0 and labels[comp + '_FAKE'] is not None:
            raise ValueError('존재하지 않는 성분의 FAKE 라벨은 None으로 두세요.')
    if any(labels[x] == 1 for x in ['VOICE_FAKE', 'MUSIC_FAKE']):
        if labels['FILE_FAKE'] == 0:
            raise ValueError('성분 FAKE=1과 FILE_FAKE=0이 충돌합니다.')
        labels['FILE_FAKE'] = 1
    return labels, entry.get('evidence', '')

def preprocess_audio(source, metadata, output_dir, cfg):
    vid = metadata['id']
    # Selection is stable across Python sessions and operating systems.
    token = f"{cfg['seed']}:{vid}"
    extension = cfg['formats'][int(stable_number(token + ':format') * len(cfg['formats']))]
    phone = stable_number(token + ':telephone') < cfg['telephone_fraction']
    with tempfile.TemporaryDirectory(dir=output_dir) as temporary:
        temp = Path(temporary)
        decoded, original_rate = decode_audio(source, temp / 'source.wav', cfg['max_source_seconds'] + 1)
        original_channels = decoded.shape[1]
        original_seconds = len(decoded) / original_rate
        if original_seconds < cfg['min_seconds'] or original_seconds > cfg['max_source_seconds'] + 0.1:
            raise SkipVideo('decoded_duration_out_of_bounds')
        duration = min(original_seconds, cfg['max_seconds'])
        offset = max(0, original_seconds - duration) * stable_number(token + ':crop')
        channels = min(2, original_channels)
        if cfg['channel_mode'] == 'mono' or phone:
            channels = 1
        elif cfg['channel_mode'] == 'mixed' and stable_number(token + ':channel') < 0.5:
            channels = 1
        # For short originals, no padding or repeat. Long Shorts yield one crop.
        frames = min(round(duration * cfg['sample_rate']), round(cfg['max_seconds'] * cfg['sample_rate']))
        normalized = temp / 'normalized.wav'
        ffmpeg_run(['-i', source, '-ss', f'{offset:.8f}', '-t', f'{duration:.8f}',
                    '-map', '0:a:0', '-vn', '-ac', channels, '-ar', cfg['sample_rate'],
                    '-c:a', 'pcm_s16le', normalized])
        x, sr = sf.read(normalized, dtype='int16', always_2d=True)
        x = x[:frames]
        if len(x) < cfg['min_seconds'] * sr:
            raise SkipVideo('clip_too_short_after_resampling')
        sf.write(normalized, x, sr, subtype='PCM_16')
        # Exact decoded-content hash before codec/telephone transforms.
        mono_raw = ffmpeg_run(['-i', normalized, '-ac', '1', '-ar', '16000',
                              '-c:a', 'pcm_s16le', '-f', 's16le', 'pipe:1'], capture=True)
        clip_hash = hashlib.sha256(mono_raw).hexdigest()
        source_pcm = ffmpeg_run(['-i', source, '-map', '0:a:0', '-t', cfg['max_source_seconds'] + 1,
                                '-ac', '1', '-ar', '16000', '-c:a', 'pcm_s16le', '-f', 's16le', 'pipe:1'], capture=True)
        pcm_hash = hashlib.sha256(source_pcm).hexdigest()
        if phone:
            telephone = temp / 'phone.wav'
            ffmpeg_run(['-i', normalized, '-af', 'highpass=f=300,lowpass=f=3400',
                        '-ar', '8000', '-ac', '1', '-c:a', 'pcm_mulaw', telephone])
            ffmpeg_run(['-i', telephone, '-ar', '16000', '-c:a', 'pcm_s16le', normalized])
        path = Path(output_dir) / f'{vid}.{extension}'
        codecs = {'flac': ['-c:a', 'flac', '-compression_level', '5'],
                  'wav': ['-c:a', 'pcm_s16le'], 'mp3': ['-c:a', 'libmp3lame', '-b:a', '64k']}
        ffmpeg_run(['-i', normalized, '-map_metadata', '-1', '-ar', '16000', '-ac', channels,
                    *codecs[extension], path])
        audio, actual_rate = decode_audio(path, temp / 'verify.wav')
        actual_duration = len(audio) / actual_rate
        if actual_rate != 16000 or audio.shape[1] not in (1, 2) or not 4 <= actual_duration <= 60:
            path.unlink(missing_ok=True)
            raise SkipVideo(f'output_spec_mismatch: {actual_rate}, {actual_duration}, {audio.shape[1]}')
        if not np.isfinite(audio).all() or np.max(np.abs(audio)) < 1e-7:
            path.unlink(missing_ok=True)
            raise SkipVideo('invalid_or_digital_silence')
        rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
        labels, label_evidence = labels_for(vid, cfg)
        group = metadata.get('channel_id') or metadata['shorts_tab']
        split_score = stable_number(f'{cfg["seed"]}:split:{group}')
        split = 'train' if split_score < 0.9 else ('validation' if split_score < 0.95 else 'test')
        record = dict(metadata)
        record.update(ID='YT_' + vid, audio_member='audio/' + path.name,
            sample_rate=actual_rate, channels=audio.shape[1], duration=actual_duration,
            original_sample_rate=original_rate, original_channels=original_channels,
            decoded_source_duration=original_seconds, crop_start_seconds=offset,
            crop_requested_seconds=duration, output_format=extension,
            telephone_simulated=phone, telephone_recipe='300-3400Hz+8kHz-mulaw+16kHz' if phone else None,
            pcm_sha256=pcm_hash, clip_pcm_sha256=clip_hash,
            dedup_basis='full_source_decoded_mono_16k_pcm_s16le',
            file_sha256=digest_file(path), file_bytes=path.stat().st_size,
            rms_dbfs=20 * math.log10(max(rms, 1e-12)),
            clipping_fraction=float(np.mean(np.abs(audio) >= 0.999)),
            source_hypothesis='ai_music_candidate', label_status='reviewed' if label_evidence else 'weak_unverified',
            labels=labels, label_evidence=label_evidence,
            weak_labels={'FILE_FAKE': 1, 'MUSIC_FAKE': 1, 'MUSIC_PRESENT': 1,
                         'VOICE_FAKE': None, 'VOICE_PRESENT': None},
            split_group=group, suggested_split=split, pipeline_version=PIPELINE_VERSION)
        return path, record

## 7. 수집 실행·보고서·파일 재검증 모듈

`manifest.csv`에는 연동 폴더 저장이 완료된 샘플만 포함합니다. `saved_unique_samples`는 이 로컬 저장 완료 수이고, `cloud_sync_verified`는 항상 `false`로 기록합니다. 확정 학습 라벨과 약한 추정, 채널 기준 제안 분할, 소스와 전처리 기록을 유지합니다.


In [ ]:
def prepare_candidates(ds, pilot=True):
    original = ds.cfg
    if pilot:
        ds.cfg = {**original, 'search_results_per_query': 25, 'max_channels': 20,
                  'shorts_per_channel': 100, 'max_candidates': 2000}
    try:
        discover_channels(ds)
        enumerate_shorts(ds)
    finally:
        ds.cfg = original
        export_reports(ds)
    print('후보 준비 완료:', ds.status())

def collect(ds, session_limit=None):
    cfg = ds.cfg
    session_limit = session_limit or cfg['session_max_new_samples']
    started, added, attempts, consecutive_errors = time.monotonic(), 0, 0, 0
    reason = 'session_limit'
    try:
        ds.recover()
        while added < session_limit:
            if ds.total_samples() >= cfg['target_unique_videos']:
                reason = 'target_reached'
                break
            if time.monotonic() - started >= cfg['session_max_seconds']:
                reason = 'session_time_limit'
                break
            if attempts >= cfg['session_max_attempts']:
                reason = 'session_attempt_limit'
                break
            if shutil.disk_usage(ds.root).free < cfg['min_free_local_gb'] * 1024**3:
                reason = 'local_disk_low'
                break
            row = ds.db.execute("SELECT * FROM videos WHERE status IN ('pending','error') AND attempts<? "
                                'ORDER BY attempts,id LIMIT 1', (cfg['max_attempts_per_video'],)).fetchone()
            if row is None:
                reason = 'candidates_exhausted'
                break
            ds.db.execute("UPDATE videos SET status='processing',attempts=attempts+1 WHERE id=?", (row['id'],))
            ds.db.commit()
            attempts += 1
            path, record = None, None
            try:
                with tempfile.TemporaryDirectory(dir=ds.raw) as temporary:
                    source, metadata = download_one(ds, row, Path(temporary))
                    path, record = preprocess_audio(source, metadata, ds.stage, cfg)
            except SkipVideo as exc:
                ds.db.execute("UPDATE videos SET status='skipped',error=? WHERE id=?", (str(exc)[:1800], row['id']))
                consecutive_errors = 0
            except Exception as exc:
                ds.db.execute("UPDATE videos SET status='error',error=? WHERE id=?", (str(exc)[:1800], row['id']))
                ds.db.commit()
                handle_network_error(exc)
                consecutive_errors += 1
                print(f'실패 {row["id"]}: {str(exc)[:200]}')
                if consecutive_errors >= cfg['max_consecutive_errors']:
                    raise StopCollection('연속 오류 한도 도달. 패키지/접근 환경을 확인한 후 재개하세요.') from exc
                time.sleep(min(30, 2 ** consecutive_errors))
            else:
                # Only content-processing errors are swallowed. Storage failures stop the run.
                duplicate = ds.db.execute('SELECT video_id FROM samples WHERE pcm_sha256=?', (record['pcm_sha256'],)).fetchone()
                if duplicate:
                    path.unlink(missing_ok=True)
                    ds.db.execute("UPDATE videos SET status='duplicate',error=? WHERE id=?", (duplicate[0], row['id']))
                else:
                    ds.db.execute('INSERT INTO samples(video_id,pcm_sha256,path,meta) VALUES (?,?,?,?)',
                        (row['id'], record['pcm_sha256'], str(path), json.dumps(record, ensure_ascii=False, allow_nan=False)))
                    ds.db.execute("UPDATE videos SET status='staged',error=NULL WHERE id=?", (row['id'],))
                    added += 1
                consecutive_errors = 0
            ds.db.commit()
            staged = ds.db.execute('SELECT COUNT(*) FROM samples WHERE shard_seq IS NULL').fetchone()[0]
            if staged >= cfg['shard_size']:
                ds.flush()
            if attempts % cfg['checkpoint_every'] == 0:
                ds.checkpoint()
                elapsed = time.monotonic() - started
                print(f'이번 실행 신규 {added:,}, 검사 {attempts:,}, {elapsed/60:.1f}분 / {ds.status()}')
    except (KeyboardInterrupt, StopCollection) as exc:
        reason = 'user_interrupt' if isinstance(exc, KeyboardInterrupt) else str(exc)
        print('수집 중단:', reason)
    except Exception:
        # In particular, don't hide Drive/quota/checksum failures behind a success summary.
        ds.db.commit()
        print('저장 또는 실행 오류로 중단했습니다. 로컬 파일을 유지합니다. 같은 데이터셋으로 재개하세요.')
        raise
    else:
        print('이번 실행 종료:', reason)
    # Also flush on graceful interruption; abrupt runtime loss recovers the last checkpoint.
    ds.flush()
    report = export_reports(ds)
    report.update(session_new_samples=added, session_attempts=attempts, session_stop_reason=reason,
                  session_elapsed_seconds=round(time.monotonic() - started, 1))
    write_json(ds.root / 'last_session.json', report)
    ds.store.put(ds.root / 'last_session.json')
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

def export_reports(ds):
    cfg = ds.cfg
    fields = ['ID','video_id','shard','shard_storage_key','audio_member','source_url','channel_id',
              'sample_rate','channels','duration','output_format','telephone_simulated','crop_start_seconds',
              'file_bytes','file_sha256','pcm_sha256','suggested_split','split_group','label_status','license_kind']
    fields += [prefix + t for prefix in ['Y_', 'MASK_', 'WEAK_'] for t in TARGETS]
    counts, total_bytes, seconds, label_counts = Counter(), 0, 0.0, Counter()
    with (ds.root / 'manifest.csv').open('w', newline='', encoding='utf-8') as f, \
         (ds.root / 'metadata.jsonl').open('w', encoding='utf-8') as raw:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for r in ds.db.execute("SELECT s.meta FROM samples s JOIN shards h ON s.shard_seq=h.seq WHERE h.status='committed' ORDER BY s.video_id"):
            m = json.loads(r[0])
            raw.write(json.dumps(m, ensure_ascii=False, allow_nan=False) + '\n')
            flat = {k: m.get(k) for k in fields}
            flat.update(video_id=m['id'], license_kind=m['rights']['kind'])
            for t in TARGETS:
                flat['Y_' + t] = m['labels'][t]
                flat['MASK_' + t] = int(m['labels'][t] is not None)
                flat['WEAK_' + t] = m['weak_labels'][t]
                label_counts[t] += flat['MASK_' + t]
            writer.writerow(flat)
            counts[f"format:{m['output_format']}"] += 1
            counts[f"channels:{m['channels']}"] += 1
            counts[f"split:{m['suggested_split']}"] += 1
            counts[f"telephone:{m['telephone_simulated']}"] += 1
            total_bytes += m['file_bytes']
            seconds += m['duration']
    for name, query in [
        ('channels.csv', 'SELECT * FROM channels ORDER BY url'),
        ('collection_log.csv', 'SELECT * FROM videos ORDER BY id'),
        ('shards.csv', "SELECT * FROM shards WHERE status='committed' ORDER BY seq")]:
        cursor = ds.db.execute(query)
        with (ds.root / name).open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([c[0] for c in cursor.description])
            writer.writerows(cursor)
    status = ds.status()
    done = status.get('done', 0)
    report = dict(updated_at=now(), target=cfg['target_unique_videos'], saved_unique_samples=done, storage_backend='desktop_sync', cloud_sync_verified=False,
        remaining=max(0, cfg['target_unique_videos'] - done), status=status,
        distribution=dict(counts), reviewed_label_counts=dict(label_counts),
        audio_gib=round(total_bytes / 1024**3, 3), total_audio_hours=round(seconds / 3600, 3),
        projected_audio_gib_at_target=round(total_bytes / max(1, done) * cfg['target_unique_videos'] / 1024**3, 1) if done else None,
        limitations=['공개 입력 규격만 일치; 비공개 평가 분포 및 전화채널 처리 미상',
                     '키워드 AI 음악 여부는 약한 추정; 확정 타깃과 구분',
                     '전체 원본을 mono 16k PCM으로 디코딩한 해시의 완전일치 중복만 제거',
                     '재인코딩/부분 중복/다른 크롭의 근접 중복은 별도 검토 필요',
                     'AI 음악만으로 전체 5개 과제의 Real/Fake와 존재 유무 균형을 충족하지 않음'])
    write_json(ds.root / 'summary.json', report)
    write_json(ds.root / 'run_config.json', cfg)
    for name in ['manifest.csv', 'metadata.jsonl', 'channels.csv', 'collection_log.csv',
                 'shards.csv', 'summary.json', 'run_config.json']:
        ds.store.put(ds.root / name)
    return report

In [ ]:
def audit_saved_shard(ds, max_items=5):
    shard = ds.db.execute("SELECT * FROM shards WHERE status='committed' ORDER BY seq DESC LIMIT 1").fetchone()
    if shard is None:
        print('아직 업로드한 shard가 없습니다.')
        return []
    results = []
    with tempfile.TemporaryDirectory(dir=ds.root) as temporary:
        temp = Path(temporary)
        tar_path = temp / shard['name']
        ds.store.get(shard['file_id'], tar_path)
        if digest_file(tar_path, 'md5') != shard['md5']:
            raise IOError('shard 체크섬 검증 실패')
        with tarfile.open(tar_path, 'r') as archive:
            for member in archive:
                if not member.isfile() or not member.name.startswith('metadata/'):
                    continue
                meta = json.load(archive.extractfile(member))
                audio_member = archive.getmember(meta['audio_member'])
                local = temp / Path(meta['audio_member']).name
                with archive.extractfile(audio_member) as src, local.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                if digest_file(local) != meta['file_sha256']:
                    raise IOError('오디오 체크섬 검증 실패')
                x, sr = decode_audio(local, temp / 'audit.wav')
                assert sr == 16000 and x.shape[1] in (1, 2) and 4 <= len(x)/sr <= 60
                results.append({'ID': meta['ID'], 'sample_rate': sr,
                                'channels': x.shape[1], 'seconds': len(x)/sr,
                                'format': meta['output_format']})
                if len(results) >= max_items:
                    break
    print('연동 폴더 파일 읽기 검증:', results)
    return results

def iter_shard_bytes(local_tar_path):
    """Read a downloaded shard without extracting 100,000 files to Drive."""
    with tarfile.open(local_tar_path, 'r') as archive:
        for member in archive:
            if member.isfile() and member.name.startswith('metadata/'):
                metadata = json.load(archive.extractfile(member))
                yield archive.extractfile(metadata['audio_member']).read(), metadata

## 8. 저장 폴더 선택 및 데이터셋 열기

`DRIVE_PARENT_PATH`가 비어 있으면 폴더 선택창에서 원래 웹 폴더에 해당하는 위치를 선택합니다. 창을 사용할 수 없는 Jupyter 환경에서는 설정 변수에 경로를 입력하세요. 선택한 부모 폴더가 실제로 존재해야 하며, 그 안에 데이터셋 하위 폴더를 생성합니다.

연동 폴더 여부나 원래 웹 폴더와의 일치 여부를 파일시스템 API로 인증하는 것은 아닙니다. 본인이 연동한 경로를 선택하세요. 잘못된 경로에서는 수집을 시작하기 전에 이 셀을 종료하고 다시 설정합니다.


In [ ]:
from importlib.metadata import version

def choose_sync_parent(value):
    selected = str(value).strip().strip('"')
    if not selected:
        try:
            import tkinter as tk
            from tkinter import filedialog
            window = tk.Tk()
            window.withdraw()
            try:
                window.attributes('-topmost', True)
                selected = filedialog.askdirectory(parent=window, mustexist=True,
                    title='지정한 Google Drive 웹 폴더에 해당하는 PC 폴더 선택')
            finally:
                window.destroy()
        except Exception as exc:
            raise RuntimeError('폴더 선택창을 열 수 없습니다. DRIVE_PARENT_PATH에 경로를 입력하세요.') from exc
    if not selected:
        raise ValueError('선택을 취소했습니다. DRIVE_PARENT_PATH를 입력하거나 이 셀을 다시 실행하세요.')
    return Path(selected).expanduser().resolve(strict=True)

DRIVE_PARENT_PATH = str(choose_sync_parent(DRIVE_PARENT_PATH))
CFG['drive_parent_path'] = DRIVE_PARENT_PATH
if 'dataset' in globals():
    dataset.db.close()
if 'store' in globals():
    store.close()
store = SyncFolderStore(DRIVE_PARENT_PATH, CFG['dataset_name'])
try:
    dataset = Dataset(CFG, store)
except BaseException:
    store.close()
    raise
print('작업 폴더:', dataset.root)
print('대상 파일시스템 여유:', f'{store.free_bytes()/1024**3:.1f} GiB (클라우드 잔여 용량 검증값 아님)')
environment = {
    'created_at': now(), 'python': sys.version, 'python_executable': sys.executable,
    'platform': platform.platform(), 'gpu_info': GPU_INFO, 'audio_backend': 'ffmpeg_cpu',
    'packages': {p: version(p) for p in ['yt-dlp', 'soundfile', 'numpy', 'imageio-ffmpeg', 'filelock']},
    'ffmpeg': command_output([FFMPEG, '-version']).splitlines()[0],
    'deno': command_output([DENO_PATH, '--version']),
    'storage_backend': 'desktop_sync', 'cloud_sync_verified': False,
}
write_json(dataset.root / 'environment.json', environment)
store.put(dataset.root / 'environment.json')
print('현재 상태:', dataset.status())


## 9. 후보 준비

pilot에서는 채널 최대 20개·채널당 100개·전체 후보 2,000개까지만 탐색합니다. full에서는 위 설정값을 사용합니다. 파일 사용권이 확인되지 않은 후보는 다음 단계에서 사유와 함께 제외되므로, 후보 수가 곧 저장 수는 아닙니다. 소량 모드에서 수집 가능 파일이 0개여도 실제 실패/제외 원인을 보고합니다.

full 모드의 초기 채널 열거는 오래 걸릴 수 있습니다. 이미 완료한 검색/채널 탐색은 체크포인트를 재사용합니다. 후보 부족 시 검색어·seed 채널을 추가하거나 `shorts_per_channel`/`max_candidates`를 늘리고 이 셀을 다시 실행합니다.

이 버전은 플레이리스트 업로드 채널 기준으로 새 데이터셋 이름을 사용합니다. 이전의 일반 AI 음악 채널 후보와 섞지 않습니다. 검색 범위를 늘리거나 최근 업로드를 다시 확인하려면 `refresh_playlist_discovery(dataset)` 후 이 셀을 실행하세요. 기본 후보 확인에는 영상 설명 요청이 추가되어 일반 제목 검색보다 시간이 더 걸립니다.


In [ ]:
prepare_candidates(dataset, pilot=(RUN_MODE == 'pilot'))
print('후보 상태:', dataset.status())

## 10. 수집 실행

pilot은 이번 실행 신규 최대 50개, full은 **누적 최대 100,000개**를 목표로 실행합니다. 로컬 기본값은 한 번에 최대 24시간이며 필요하면 `session_max_seconds`를 변경할 수 있습니다. 24시간 안에 목표에 도달한다고 보장하지 않습니다.

중지 후 같은 설정으로 다시 열고 이 셀을 실행하면 이어집니다. 후보를 늘릴 때는 9번도 실행합니다. 파일 복사/체크섬/공간 오류는 저장 성공으로 처리하지 않고 중단하며 작업 파일을 유지합니다. 작업 중 PC가 절전 상태로 들어가면 처리가 멈출 수 있습니다.


In [ ]:
result = collect(dataset, session_limit=50 if RUN_MODE == 'pilot' else CFG['session_max_new_samples'])

## 11. 연동 폴더의 저장 파일 재검증

저장된 최근 TAR를 로컬 작업 폴더로 읽어 TAR MD5, 개별 오디오 SHA256, 16 kHz·4~60초·채널 수를 확인합니다. 스트리밍 드라이브가 파일을 캐시에서 읽을 수 있으므로 **이 검사는 클라우드 업로드 완료 검사가 아닙니다.**


In [ ]:
audit_saved_shard(dataset, max_items=5)

## 12. 저장 결과 확인과 재시도

| 파일 | 내용 |
|---|---|
| `audio-000001.tar`, … | `audio/<영상ID>.<확장자>`와 `metadata/<영상ID>.json`, 기본 500개 묶음 |
| `manifest.csv` | 학습 입력 경로, 출처, 규격, 분할, 확인 라벨/마스크, 약한 추정 |
| `metadata.jsonl` | 영상 설명·라이선스 근거·생성 근거·크롭·전처리·해시 |
| `channel_screening.csv` | 채널 선별 결과, 대표 플레이리스트와 생성 근거 |
| `channels.csv` | 발견한 채널과 근거, 채널 수집 진행 상태 |
| `collection_log.csv` | 영상별 완료/스킵/실패/중복 및 오류 사유 |
| `shards.csv` | 저장 완료 TAR의 파일 이름·크기·MD5 |
| `state.sqlite` | 중단·재개용 체크포인트 |
| `summary.json`, `last_session.json` | 누적 수량, 부족분, 크기 추정, 최근 실행 결과 |
| `run_config.json`, `environment.json` | 수집 설정, 실행 환경 버전 |

학습 시 TAR를 로컬 작업 폴더로 한 묶음씩 복사한 뒤 `iter_shard_bytes()`로 순회할 수 있습니다. `audio_member`는 TAR 내부 경로이며 Drive의 개별 파일 ID가 아닙니다. Drive에 오디오 10만 개를 각각 풀어놓을 필요는 없습니다.

권한 근거/채널을 추가했으면 설정 셀을 수정하고 `dataset.cfg = CFG`를 실행한 뒤 후보 준비 셀을 실행합니다. 이후 아래 재시도 함수를 필요할 때 호출하세요. 이미 저장된 샘플의 라벨을 수정하려면 별도 검토 파일을 만들어 영상 ID로 결합하거나 새 데이터셋을 생성하세요. 설정 변경만으로 기존 샘플 라벨이 소급 변경되지는 않습니다.

In [ ]:
print(json.dumps(dataset.status(), ensure_ascii=False, indent=2))
print('저장 폴더:', store.root)

# 검색어/이용허락 근거를 설정에 추가한 후:
# dataset.cfg = CFG
# dataset.retry_errors()
# dataset.reconsider_skipped()
# refresh_playlist_discovery(dataset)
# prepare_candidates(dataset, pilot=False)
# result = collect(dataset)

# TAR에서 바로 오디오 bytes와 메타데이터 읽기:
# for audio_bytes, metadata in iter_shard_bytes(store.root / 'audio-000001.tar'):
#     print(metadata['ID'], len(audio_bytes), metadata['labels'])
#     break

# 작업을 마치고 같은 PC의 다른 커널에서 이어서 실행하려면:
# dataset.checkpoint(force=True)
# dataset.db.close()
# store.close()


## 운영 메모

- 10만 개 달성 가능성은 확보 가능한 서로 다른 영상·허용된 이용조건·접속 환경에 따라 달라집니다. 적격 후보가 소진되면 `candidates_exhausted`와 부족분을 남기며, 복제/증강으로 채우지 않습니다.
- `rights_unverified`가 많으면 제작자로부터 대상 콘텐츠 이용허락 근거를 확보하거나, 이용조건이 확인된 채널을 추가하세요. 기존 후보는 `reconsider_skipped()`로 재검토합니다.
- `no_video_or_playlist_channel_ai_music_evidence`가 많으면 영상 설명을 확인하세요. AI 영상·AI 커버 보컬만 생성하고 반주는 실제인 경우도 있어, 수집 후보를 그대로 AI 반주 정답으로 쓰면 안 됩니다.
- 429/403/봇 확인은 자동 우회를 하지 않고 중단합니다. 접근 가능한 환경과 서비스 이용조건을 확인한 뒤 재개하세요. 무한 재시도나 다중 런타임으로 수량을 보충하지 않습니다.
- 무료 Drive 기본 용량만으로 10만 개 오디오가 모두 들어간다고 가정하지 않습니다. 실제 샘플을 저장한 후 `projected_audio_gib_at_target`와 Drive 앱/웹에서 확인한 계정 용량을 비교하세요. 공유 드라이브에는 별도 관리 한도도 적용될 수 있습니다.
- 목표에 도달한 뒤에도 확정 라벨, 근접 중복, 생성기/곡 단위 누수, Real/Fake 균형을 검토해야 합니다. 이 노트북은 수집 도구이며 추론 모델/대회 제출 ZIP을 만들지는 않습니다.

### 검증 범위

Windows에서 실제 파일시스템으로 연동 폴더 저장 계층을 테스트하며, 복사 실패 후 원본 보존·재개·파일 잠금·체크섬·경로 검사와 실제 FFmpeg 오디오 변환을 검증합니다. 실제 YouTube 대량 다운로드와 Google Drive 클라우드 동기화는 실행하지 않았습니다. 이전 버전과 동일하게 이용조건·불확실한 라벨·10만 개 확보 가능성에 관한 제한이 적용됩니다.


플레이리스트 채널에 Shorts가 없으면 그 채널에서는 수집하지 않습니다. 따라서 플레이리스트 업로드 수와 수집할 수 있는 Shorts 수는 다릅니다. `channel_playlist_inferred` 자료는 채널 성격에서 추정한 후보이므로, 평가용 정답으로 사용할 때 별도 검토해야 합니다.
